# MOF Screening V3 — Colab Ready CPU Pipeline

Run the cells in order. The notebook will:

- mount Google Drive automatically;
- reuse an already extracted project if it is valid;
- otherwise find the full project ZIP in Drive, or prompt you to upload it from the PC;
- install/check dependencies;
- archive stale results before the new run;
- run all integrity checks and the corrected V3 pipeline on CPU;
- save the log and final results ZIP in Drive;
- download the final result package to the PC.

No GPU is required.


In [ ]:
# 1) Mount Drive and prepare the full project automatically
from google.colab import drive, files
from pathlib import Path
import os, shutil, zipfile, tempfile, json

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_PARENT = DRIVE_ROOT / 'MOF_Screening_V3_Project'
PROJECT = PROJECT_PARENT / 'mof_screening_v3'
EXPECTED_VERSION = '3.0.0'


def valid_project(path: Path) -> bool:
    required = [
        path / 'src' / 'run_pipeline.py',
        path / 'src' / 'config.py',
        path / 'requirements.txt',
        path / 'data' / 'raw' / 'core_uptake.csv',
        path / 'data' / 'raw' / 'core_ch4uptake_highP.csv',
    ]
    return all(p.exists() for p in required)


def find_project_root(extract_root: Path) -> Path:
    candidates = []
    for p in [extract_root] + list(extract_root.rglob('*')):
        if p.is_dir() and valid_project(p):
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError('The ZIP does not contain a valid mof_screening_v3 project.')
    candidates.sort(key=lambda p: len(p.parts))
    return candidates[0]


if valid_project(PROJECT):
    print('Reusing existing project:', PROJECT)
else:
    PROJECT_PARENT.mkdir(parents=True, exist_ok=True)

    zip_candidates = []
    preferred_names = {
        'MOF_Screening_V3_FULL_COLAB_READY.zip',
        'MOF_Screening_V3_FULL.zip',
    }
    for p in DRIVE_ROOT.rglob('*.zip'):
        if p.name in preferred_names or 'MOF_Screening_V3_FULL' in p.name:
            zip_candidates.append(p)

    if zip_candidates:
        zip_candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        source_zip = zip_candidates[0]
        print('Using project ZIP from Drive:', source_zip)
    else:
        print('Full project ZIP was not found in Drive.')
        print('Select MOF_Screening_V3_FULL_COLAB_READY.zip or MOF_Screening_V3_FULL.zip from the PC.')
        uploaded = files.upload()
        uploaded_zips = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
        if not uploaded_zips:
            raise FileNotFoundError('No ZIP file was uploaded.')
        source_zip = uploaded_zips[0]
        drive_copy = DRIVE_ROOT / source_zip.name
        shutil.copy2(source_zip, drive_copy)
        source_zip = drive_copy
        print('Copied uploaded ZIP to Drive:', source_zip)

    temp_extract = Path(tempfile.mkdtemp(prefix='mof_v3_extract_'))
    try:
        with zipfile.ZipFile(source_zip, 'r') as zf:
            zf.extractall(temp_extract)
        extracted_project = find_project_root(temp_extract)

        if PROJECT_PARENT.exists():
            shutil.rmtree(PROJECT_PARENT)
        PROJECT_PARENT.mkdir(parents=True, exist_ok=True)
        shutil.copytree(extracted_project, PROJECT)
    finally:
        shutil.rmtree(temp_extract, ignore_errors=True)

    if not valid_project(PROJECT):
        raise RuntimeError('Project extraction failed validation.')
    print('Project extracted to:', PROJECT)

# Validate pipeline version without importing third-party packages
config_text = (PROJECT / 'src' / 'config.py').read_text(encoding='utf-8')
if f'PIPELINE_VERSION = "{EXPECTED_VERSION}"' not in config_text:
    raise RuntimeError(f'Unexpected pipeline version. Expected {EXPECTED_VERSION}.')

os.chdir(PROJECT)
print('\nPROJECT READY')
print('Project path:', PROJECT)
print('Drive mounted:', DRIVE_ROOT.exists())
print('Pipeline version:', EXPECTED_VERSION)


In [ ]:
# 2) Install/check Python dependencies
import subprocess, sys
from pathlib import Path

requirements = PROJECT / 'requirements.txt'
if not requirements.exists():
    raise FileNotFoundError(requirements)

cmd = [
    sys.executable, '-m', 'pip', 'install',
    '--disable-pip-version-check', '-q',
    '-r', str(requirements)
]
print('Installing/checking dependencies...')
subprocess.run(cmd, check=True)
print('Dependencies ready.')


In [ ]:
# 3) Preflight validation: data, imports, project integrity and model smoke test
import os, subprocess, sys, json
from pathlib import Path

os.chdir(PROJECT)
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # explicitly use CPU

required_raw = [
    'core_ch4uptake_highP.csv',
    'core_density.csv',
    'core_di.csv',
    'core_logKH_CH4.csv',
    'core_logKH_CO2.csv',
    'core_uptake.csv',
]
raw_dir = PROJECT / 'data' / 'raw'
missing = [name for name in required_raw if not (raw_dir / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing raw data files: {missing}')

print('Raw files verified:', len(required_raw))
subprocess.run([sys.executable, 'src/run_checks.py'], check=True)
subprocess.run([sys.executable, 'tests/test_model_smoke.py'], check=True)
print('\nALL PREFLIGHT CHECKS PASSED')


In [ ]:
# 4) Archive stale outputs, create a clean result directory, and run the full pipeline
import os, shutil, subprocess, sys
from datetime import datetime
from pathlib import Path

os.chdir(PROJECT)
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

results_dir = PROJECT / 'results'
backup_root = PROJECT / 'previous_result_backups'
log_path = PROJECT / 'full_v3_run.log'

# Back up any non-empty old results so stale files cannot contaminate the new run.
if results_dir.exists() and any(p.is_file() for p in results_dir.rglob('*')):
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_root.mkdir(parents=True, exist_ok=True)
    backup_dir = backup_root / f'results_before_run_{stamp}'
    shutil.copytree(results_dir, backup_dir)
    print('Previous results archived to:', backup_dir)

if results_dir.exists():
    shutil.rmtree(results_dir)
for sub in ['tables', 'figures', 'models', 'audits']:
    (results_dir / sub).mkdir(parents=True, exist_ok=True)

if log_path.exists():
    log_path.unlink()

print('\nStarting corrected V3 CPU pipeline...')
print('Live log:', log_path)

process = subprocess.Popen(
    [sys.executable, '-u', 'src/run_pipeline.py'],
    cwd=str(PROJECT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)

with log_path.open('w', encoding='utf-8') as log_file:
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Pipeline failed with return code {return_code}. Check {log_path}')

print('\nFULL V3 PIPELINE FINISHED SUCCESSFULLY')


In [ ]:
# 5) Verify final outputs, save a Drive backup, and download everything to the PC
import json, shutil, os
from pathlib import Path
from datetime import datetime
from google.colab import files

results_dir = PROJECT / 'results'
required_outputs = [
    results_dir / 'audits' / 'run_manifest.json',
    results_dir / 'audits' / 'split_audit.csv',
]
missing_outputs = [str(p) for p in required_outputs if not p.exists()]
if missing_outputs:
    raise FileNotFoundError(f'Required final outputs are missing: {missing_outputs}')

all_files = [p for p in results_dir.rglob('*') if p.is_file()]
png_files = sorted(results_dir.rglob('*.png'))
pdf_files = sorted(results_dir.rglob('*.pdf'))
csv_files = sorted(results_dir.rglob('*.csv'))
model_files = sorted((results_dir / 'models').rglob('*')) if (results_dir / 'models').exists() else []
model_files = [p for p in model_files if p.is_file()]

print('=' * 70)
print('FINAL OUTPUT VERIFICATION')
print('=' * 70)
print('Total result files:', len(all_files))
print('CSV files:', len(csv_files))
print('PNG figures:', len(png_files))
print('PDF figures:', len(pdf_files))
print('Saved model files:', len(model_files))

if not png_files:
    raise RuntimeError('No PNG figures were generated.')

# Include results, run log, requirements and source code needed for reproducibility.
package_root = Path('/content/MOF_Screening_V3_Final_Package')
if package_root.exists():
    shutil.rmtree(package_root)
package_root.mkdir(parents=True)

shutil.copytree(results_dir, package_root / 'results')
shutil.copy2(PROJECT / 'full_v3_run.log', package_root / 'full_v3_run.log')
shutil.copy2(PROJECT / 'requirements.txt', package_root / 'requirements.txt')
shutil.copytree(PROJECT / 'src', package_root / 'src')
shutil.copytree(PROJECT / 'tests', package_root / 'tests')

# Add a readable package manifest.
manifest_txt = package_root / 'PACKAGE_CONTENTS.txt'
with manifest_txt.open('w', encoding='utf-8') as f:
    for p in sorted(package_root.rglob('*')):
        if p.is_file():
            f.write(f'{p.relative_to(package_root)} | {p.stat().st_size} bytes\n')

zip_path = Path(shutil.make_archive(
    '/content/MOF_Screening_V3_Final_Package',
    'zip',
    root_dir=package_root,
))

drive_zip = DRIVE_ROOT / 'MOF_Screening_V3_Final_Package.zip'
shutil.copy2(zip_path, drive_zip)

print('\nDrive backup:', drive_zip)
print('PC download file:', zip_path)
files.download(str(zip_path))
